# TP1 - LABORATORIO DE MODELOS ANALÍTICOS (ISTEA 2026)
### Dataset: Online Retail II (UCI)

In [1]:
import pandas as pd
import numpy as np

# 1. DESCARGA Y EXTRACCIÓN
print("Descargando dataset...")
!wget -q -O r.zip "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
!unzip -o -q r.zip

Descargando dataset...


In [2]:
# 2. CARGA DE LAS DOS HOJAS (2009-2010 y 2010-2011)
print("Cargando hojas de cálculo (esto puede demorar unos 40 segundos)...")
hojas = pd.read_excel("online_retail_II.xlsx", sheet_name=None)
tx_raw = pd.concat(hojas.values(), ignore_index=True)

# Guardar en CSV local para lecturas rápidas posteriores sin volver a procesar el Excel
tx_raw.to_csv("online_retail_consolidado.csv", index=False)
print(f"Dataset bruto cargado: {tx_raw.shape[0]:,} filas y {tx_raw.shape[1]} columnas.")

Cargando hojas de cálculo (esto puede demorar unos 40 segundos)...
Dataset bruto cargado: 1,067,371 filas y 8 columnas.


In [3]:
# 3. CONFORMACIÓN DE LA VARIABLE MONTO
tx_raw["monto"] = tx_raw["Quantity"] * tx_raw["Price"]

In [4]:
# 4. VERIFICACIÓN DE GRANULARIDAD Y CATÁLOGO
print("\n--- 1. VERIFICACIÓN DE CATÁLOGO ---")
print(f"Rango de fechas: desde {tx_raw['InvoiceDate'].min()} hasta {tx_raw['InvoiceDate'].max()}")
print(f"Invoices distintos: {tx_raw['Invoice'].nunique():,}")
print(f"Líneas promedio por Invoice: {len(tx_raw) / tx_raw['Invoice'].nunique():.2f}")
print(f"Customer IDs únicos (no nulos): {tx_raw['Customer ID'].dropna().nunique():,}")
print(f"Compras/líneas promedio por Customer ID: {tx_raw['Customer ID'].dropna().count() / tx_raw['Customer ID'].dropna().nunique():.2f}")


--- 1. VERIFICACIÓN DE CATÁLOGO ---
Rango de fechas: desde 2009-12-01 07:45:00 hasta 2011-12-09 12:50:00
Invoices distintos: 53,628
Líneas promedio por Invoice: 19.90
Customer IDs únicos (no nulos): 5,942
Compras/líneas promedio por Customer ID: 138.74


In [5]:
# 5. DIAGNÓSTICO DE CALIDAD
print("\n--- 2. DIAGNÓSTICO DE CALIDAD Y PUNTOS DE DEGRADACIÓN ---")
duplicados = tx_raw.duplicated().sum()
nulos_cliente = tx_raw["Customer ID"].isna().sum()
nulos_desc = tx_raw["Description"].isna().sum()
cancelaciones = tx_raw["Invoice"].astype(str).str.startswith("C").sum()
cant_negativa = (tx_raw["Quantity"] <= 0).sum()
precio_cero = (tx_raw["Price"] <= 0).sum()

print(f"1. Filas exactamente duplicadas: {duplicados:,} ({duplicados/len(tx_raw):.2%})")
print(f"2. Filas sin Customer ID (anónimos): {nulos_cliente:,} ({nulos_cliente/len(tx_raw):.2%})")
print(f"3. Filas sin Description: {nulos_desc:,} ({nulos_desc/len(tx_raw):.2%})")
print(f"4. Filas que son cancelaciones (Invoice empieza con 'C'): {cancelaciones:,} ({cancelaciones/len(tx_raw):.2%})")
print(f"5. Filas con Quantity <= 0: {cant_negativa:,} ({cant_negativa/len(tx_raw):.2%})")
print(f"6. Filas con Price <= 0: {precio_cero:,} ({precio_cero/len(tx_raw):.2%})")


--- 2. DIAGNÓSTICO DE CALIDAD Y PUNTOS DE DEGRADACIÓN ---
1. Filas exactamente duplicadas: 34,335 (3.22%)
2. Filas sin Customer ID (anónimos): 243,007 (22.77%)
3. Filas sin Description: 4,382 (0.41%)
4. Filas que son cancelaciones (Invoice empieza con 'C'): 19,494 (1.83%)
5. Filas con Quantity <= 0: 22,950 (2.15%)
6. Filas con Price <= 0: 6,207 (0.58%)


In [6]:
# 6. APLICACIÓN DE LA BITÁCORA DE LIMPIEZA
# Criterio: Datos para modelado de comportamiento longitudinal de clientes
print("\n--- 3. EJECUCIÓN DE LA LIMPIEZA (BITÁCORA) ---")
# Paso A: Eliminar duplicados exactos
tx = tx_raw.drop_duplicates().copy()
print(f"Tras quitar duplicados: {len(tx):,} filas.")

# Paso B: Excluir compras anónimas (no tienen tracking posible)
tx_clientes = tx[tx["Customer ID"].notna()].copy()
tx_clientes["Customer ID"] = tx_clientes["Customer ID"].astype(int).astype(str)
print(f"Tras filtrar compras con Customer ID: {len(tx_clientes):,} filas.")

# Paso C: Aislar transacciones positivas válidas
tx_compras = tx_clientes[(tx_clientes["Quantity"] > 0) & (tx_clientes["Price"] > 0)].copy()
# Excluir facturas marcadas como cancelación explícita
tx_compras = tx_compras[~tx_compras["Invoice"].astype(str).str.startswith("C")]
print(f"Tabla depurada de compras comerciales efectivas: {len(tx_compras):,} filas.")


--- 3. EJECUCIÓN DE LA LIMPIEZA (BITÁCORA) ---
Tras quitar duplicados: 1,033,036 filas.
Tras filtrar compras con Customer ID: 797,885 filas.
Tabla depurada de compras comerciales efectivas: 779,425 filas.


In [7]:
# 7. CONSTRUCCIÓN DE LA TABLA DE CLIENTES (RFM BÁSICO)
ID, FECHA, VALOR = "Customer ID", "InvoiceDate", "monto"
CORTE = tx_compras[FECHA].max()

clientes = tx_compras.groupby(ID).agg(
    frecuencia=(ID, "size"),
    compras_unicas=("Invoice", "nunique"),
    monto_total=(VALOR, "sum"),
    ticket_promedio=(VALOR, "mean"),
    ultima_compra=(FECHA, "max")
)

clientes["recencia_dias"] = (CORTE - clientes["ultima_compra"]).dt.days

print("\n--- 4. TABLA DE PERSONAS RESULTANTE ---")
print(f"Cantidad total de clientes evaluados: {len(clientes):,}")
print(f"Fecha de corte del dataset: {CORTE.date()}")
display(clientes.head())


--- 4. TABLA DE PERSONAS RESULTANTE ---
Cantidad total de clientes evaluados: 5,878
Fecha de corte del dataset: 2011-12-09


,frecuencia,compras_unicas,monto_total,ticket_promedio,ultima_compra,recencia_dias
Customer ID,,,,,,
12346,34,12,77556.46,2281.072353,2011-01-18 10:01:00,325
12347,222,8,4921.53,22.169054,2011-12-07 15:52:00,1
12348,51,5,2019.40,39.596078,2011-09-25 13:13:00,74
12349,175,4,4428.69,25.306800,2011-11-21 09:51:00,18
12350,17,1,334.40,19.670588,2011-02-02 16:01:00,309


In [8]:
# 8. RESPUESTA AL ESTADO ACTUAL DEL NEGOCIO (INACTIVOS A 90 DÍAS)
dormidos = clientes[clientes["recencia_dias"] > 90]
print("\n--- 5. ESTADO DE RETENCIÓN ---")
print(f"Clientes sin compras en los últimos 90 días: {len(dormidos):,} ({len(dormidos)/len(clientes):.1%})")
print(f"Facturación histórica en riesgo: £ {dormidos['monto_total'].sum():,.2f} ({dormidos['monto_total'].sum() / clientes['monto_total'].sum():.1%} del total)")


--- 5. ESTADO DE RETENCIÓN ---
Clientes sin compras en los últimos 90 días: 2,985 (50.8%)
Facturación histórica en riesgo: £ 3,384,937.11 (19.5% del total)
